# **Libraries**

In [ ]:
# --------------------------------------------------------
#  Import Libraries
# --------------------------------------------------------
import base64, json, time, msal, requests, pyodbc
from typing import Dict, List, Optional
import notebookutils.credentials as cred

# **Azure Key Vault**

In [ ]:
# --------------------------------------------------------
# Define Azure Key Vault
# --------------------------------------------------------
key_vault_name = "hcm-prod-kv"
key_vault_url = f"https://{key_vault_name}.vault.azure.net/"

# --------------------------------------------------------
# Get Azure Key Vault Service Principal Credentials
# --------------------------------------------------------
client_name = "HCM-SPN-Fabric"
tenant_id = cred.getSecret(key_vault_url,"FabricTenantId")
client_id = cred.getSecret(key_vault_url,"FabricClientId")
client_secret = cred.getSecret(key_vault_url,"FabricClientSecret")

# **Functions**

## **Access token function**

In [ ]:
# --------------------------------------------------------
#  Get access token function
# --------------------------------------------------------
def get_access_token(tenant_id, client_id, client_secret):
    """
    Retrieve a Fabric API access token using a service principal.
    Returns the token string, or raises an exception with a clear message.
    """

    # token_provider_uri = f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
    token_provider_uri = f"https://login.microsoftonline.com/{tenant_id}/oauth2/v2.0/token"

    headers = {'Content-Type': 'application/x-www-form-urlencoded'}

    body = {
        'grant_type': 'client_credentials',
        'client_id': client_id,
        'client_secret': client_secret,
        'resource': 'https://api.fabric.microsoft.com',
        'scope': 'https://api.fabric.microsoft.com/.default'
    }

    try:
        response = requests.post(token_provider_uri, data=body, headers=headers)
        response.raise_for_status()
    except requests.exceptions.HTTPError as http_err:
        # Fabric loves giving vague 401s, so let's make it obvious
        raise RuntimeError(
            f"Failed to retrieve access token. "
            f"HTTP error: {http_err}. "
            f"Check tenant ID, client ID, secret, and SP permissions."
        )
    except Exception as ex:
        raise RuntimeError(
            f"Unexpected error while retrieving access token: {ex}"
        )

    token = response.json().get("access_token")
    if not token:
        raise RuntimeError("Token request succeeded but no access_token was returned.")

    print("Access token successfully retrieved.")
    return token

# --------------------------------------------------------
#  Get capacity id function
# --------------------------------------------------------
def get_capacity_id(access_token: str, workspace_id: str) -> str:
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.get(endpoint, headers=headers)
    response.raise_for_status()

    data = response.json()

    # capacity ID shows up as: data["capacityId"]
    return data.get("capacityId")

# --------------------------------------------------------
#  Get Fabric Item Id by display name and type
# --------------------------------------------------------
def get_item_id(access_token: str, item_name: str, item_type: str = None) -> str:
    """
    Retrieve the ID of a Fabric item by its display name (and optional type).
    """
    workspace_id = spark.conf.get("trident.workspace.id")
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.get(endpoint, headers=headers)
    response.raise_for_status()

    items = response.json().get("value", [])

    for item in items:
        if item["displayName"] == item_name:
            if item_type is None or item["type"] == item_type:
                return item["id"]

    raise ValueError(f"Item '{item_name}' (type={item_type}) not found in workspace.")

# **Operation**

## **Get access token**

In [ ]:
# --------------------------------------------------------
#  Get access token
# --------------------------------------------------------
access_token = get_access_token(tenant_id, client_id, client_secret)

# --------------------------------------------------------
#  Get current Workspace Id
# --------------------------------------------------------
workspace_id = spark.conf.get("trident.workspace.id")
print(f"Workspace Id: {workspace_id}")

# --------------------------------------------------------
#  Get current Capacity Id
# --------------------------------------------------------
capacity_id = get_capacity_id(access_token, workspace_id)
print("Capacity Id:", capacity_id)